# Notebook 3: Semantic Cache Evaluation
## SmartCache ML: Persistent SQLite Cache, Latency Breakdown & Isolation

### 1. Problem Overview
A production semantic cache requires more than in-memory similarity:
1. **Durability**: Entries must survive service restarts (backed by SQLite).
2. **Latency transparency**: We must isolate embedding generation time from similarity search time.
3. **Multi-tenancy / Privacy**: Queries must be partitioned by `namespace` so private user data is never cross-served.
4. **Lifecycle management**: Cache entries must respect TTL expiration and LRU eviction limits.
5. **Freshness guarantees**: Dynamic information (real-time timestamps, fluctuating prices) must be flagged non-cacheable.

---
### Learning Objectives
By the end of this notebook, you will be able to:
- Implement a persistent SQLite-backed semantic cache with vectorized NumPy similarity search.
- Dissect cache lookup latency into embedding generation vs vector search components.
- Verify multi-tenant privacy via namespace isolation.
- Demonstrate TTL expiration and LRU capacity eviction.
- Formulate safety policies regarding what should and should not be cached.

In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from smartcache.config import settings
from smartcache.cache import SemanticCache, ExactMatchCache
from smartcache.cost_tracker import CostTracker

# Use temporary notebook cache db
notebook_db = settings.project_root / "data" / "notebook_cache.db"
if notebook_db.exists():
    notebook_db.unlink()

cache = SemanticCache(db_path=notebook_db, similarity_threshold=0.85, max_cache_size=100)
print(f"Initialized SemanticCache with SQLite db: {notebook_db}")

### 2. Dissecting Lookup Latency: Embedding vs Vector Search
Let's measure the exact latency breakdown between:
- Embedding model encoding (CPU tensor forward pass).
- Matrix-vector cosine search over cached items.

In [ ]:
# Populate cache with 50 diverse items
df = pd.read_csv(settings.data_raw_path / "benchmark_dataset.csv")
canonicals = df[df['query_role'] == 'canonical'].head(50)

for _, row in canonicals.iterrows():
    cache.store(row['query_text'], row['expected_answer'], is_cacheable=row['is_cacheable'])

print(f"Cached entries: {len(cache._ids)}")

# Profile 20 lookups
embed_times = []
search_times = []
total_times = []

for _, row in df.head(20).iterrows():
    res = cache.lookup(row['query_text'])
    embed_times.append(res.embedding_latency * 1000.0)
    search_times.append(res.search_latency * 1000.0)
    total_times.append(res.total_lookup_latency * 1000.0)

print(f"Mean Embedding Latency:     {np.mean(embed_times):.2f} ms")
print(f"Mean Vector Search Latency: {np.mean(search_times):.3f} ms")
print(f"Mean Total Lookup Latency:  {np.mean(total_times):.2f} ms")
print(f"Vector search represents {np.mean(search_times) / np.mean(total_times) * 100:.2f}% of total cache lookup time.")

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(['Embedding Forward Pass', 'Vector Dot-Product Search'], [np.mean(embed_times), np.mean(search_times)], color=['#1f77b4', '#2ca02c'])
plt.ylabel("Latency (milliseconds)")
plt.title("Cache Lookup Latency Breakdown")
plt.yscale('log')
plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.show()

### 3. Namespace Isolation for Multi-Tenancy and Privacy
To prevent confidential information leaking across tenants or users, every entry is scoped to a `namespace`.

In [ ]:
# Scenario: User Alice stores private financial inquiry
cache.store(
    query="What is my current bank account balance?",
    response="Account ending in 4102 has $8,420.00.",
    namespace="tenant_user_alice",
    is_cacheable=True
)

# User Bob asks the same question in his namespace
res_bob = cache.lookup("What is my current bank account balance?", namespace="tenant_user_bob")
print(f"Bob's lookup hit status: {res_bob.hit}")
print(f"Did Bob get Alice's response? {res_bob.response}")

# Alice queries her own namespace
res_alice = cache.lookup("What is my current bank account balance?", namespace="tenant_user_alice")
print(f"Alice's lookup hit status: {res_alice.hit}")
print(f"Alice's cached response: {res_alice.response}")

### 4. TTL Expiration & Dynamic Information Safety
Responses to queries regarding volatile information (time, stock prices, live sensor telemetry) must not be cached.

In [ ]:
# Volatile query flagged as non-cacheable
volatile_query = "What is the live stock price of Apple right now?"
stored = cache.store(volatile_query, "AAPL is $225.40", is_cacheable=False)
print(f"Attempted to store non-cacheable query -> Stored successfully? {stored}")

res_volatile = cache.lookup(volatile_query)
print(f"Lookup hit for non-cacheable query: {res_volatile.hit}")

### 5. Interview Concepts Checklist
1. **Flat NumPy Search vs Vector Databases (HNSW / FAISS)**: For caches under 50,000 vectors, in-memory NumPy matrix multiplication takes < 1 ms on CPU and avoids network hop latency. Dedicated vector DBs become necessary when cache size exceeds memory or requires sharding.
2. **Multi-Tenant Cache Partitioning**: Why logical namespacing is critical in enterprise RAG and LLM systems to avoid GDPR / HIPAA compliance breaches.
3. **Cache Eviction Policies**: Trade-offs between Least Recently Used (LRU), Least Frequently Used (LFU), and Time-To-Live (TTL) expiration.